# Deep Learning Dyslexia Risk Prediction
**Based on:**
1. Rello et al. (2020) — Original PLOS ONE gamified test (Random Forest, ~80%)
2. Alluhaidan et al. (2024) — Deep Learning improvement (ANN, ~97%)
3. English Adaptation paper — Question design for English

**Pipeline:** IQR outlier removal → Feature engineering → SMOTE → Multiple models → Ensemble

In [ ]:
import pandas as pd, numpy as np, os, warnings, joblib
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
from sklearn.preprocessing import StandardScaler, RobustScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
    HAS_XGB = True
except: HAS_XGB = False
try:
    import lightgbm as lgb
    HAS_LGB = True
except: HAS_LGB = False

MEASURES = ['Clicks','Hits','Misses','Score','Accuracy','Missrate']
DEMOGRAPHIC = ['Gender','Nativelang','Otherlang','Age']
TARGET = 'Dyslexia'
N_QUESTIONS = 32
DATA_DIR = 'Data'
np.random.seed(42)
print('✅ Setup complete. XGBoost:', HAS_XGB, '| LightGBM:', HAS_LGB)

## 1. Load & Preprocess Data

In [ ]:
def load_data(path):
    try:
        df = pd.read_csv(path, sep=';', encoding='utf-8')
        if len(df.columns) < 10:
            df = pd.read_csv(path, sep=',', encoding='utf-8')
    except:
        df = pd.read_csv(path, sep=',', encoding='utf-8')
    if 'Gender' in df.columns:
        df['Gender'] = df['Gender'].map({'Male':1,'Female':0}).fillna(0).astype(int)
    for c in ['Nativelang','Otherlang','Dyslexia']:
        if c in df.columns:
            df[c] = df[c].map({'Yes':1,'No':0}).fillna(0).astype(int)
    return df

desktop = load_data(os.path.join(DATA_DIR, 'Dyt-desktop.csv'))
tablet  = load_data(os.path.join(DATA_DIR, 'Dyt-tablet.csv'))
print(f'Desktop: {desktop.shape[0]} rows, {desktop.shape[1]} cols')
print(f'Tablet:  {tablet.shape[0]} rows, {tablet.shape[1]} cols')
print(f'\nClass distribution (Desktop):')
print(desktop[TARGET].value_counts().rename({0:'Non-Dyslexia',1:'Dyslexia'}))
print(f'\nImbalance ratio: {desktop[TARGET].value_counts()[0]/desktop[TARGET].value_counts()[1]:.1f}:1')

## 2. Outlier Removal (IQR Method)
**From Alluhaidan et al.:** Outliers detected via boxplots and removed using IQR.
This removes participants with erratic click patterns or inconsistent responses.

In [ ]:
def remove_outliers_iqr(df, columns, factor=1.5):
    mask = pd.Series(True, index=df.index)
    for col in columns:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        if IQR > 0:
            mask &= (df[col] >= Q1 - factor*IQR) & (df[col] <= Q3 + factor*IQR)
    return df[mask].copy()

perf_cols = [c for c in desktop.columns if c not in DEMOGRAPHIC + [TARGET]]
desk_clean = remove_outliers_iqr(desktop, perf_cols, factor=2.0)
print(f'Before outlier removal: {len(desktop)}')
print(f'After outlier removal:  {len(desk_clean)} (removed {len(desktop)-len(desk_clean)})')
print(f'Class distribution after:')
print(desk_clean[TARGET].value_counts().rename({0:'Non-Dyslexia',1:'Dyslexia'}))

## 3. Advanced Feature Engineering
**Key insight from papers:** Raw features alone give ~80%. Adding engineered features
(per-question aggregates, cross-question statistics, behavioral patterns) boosts accuracy.

In [ ]:
def engineer_features(df):
    df = df.copy()
    for q in range(1, N_QUESTIONS+1):
        h, m, c = f'Hits{q}', f'Misses{q}', f'Clicks{q}'
        if h in df.columns and m in df.columns:
            df[f'HitMissRatio{q}'] = df[h] / (df[m] + 1)
            df[f'Engagement{q}'] = df[c] / (df[c].max() + 1)

    # Cross-question aggregates
    hit_cols = [f'Hits{q}' for q in range(1, N_QUESTIONS+1) if f'Hits{q}' in df.columns]
    miss_cols = [f'Misses{q}' for q in range(1, N_QUESTIONS+1) if f'Misses{q}' in df.columns]
    acc_cols = [f'Accuracy{q}' for q in range(1, N_QUESTIONS+1) if f'Accuracy{q}' in df.columns]
    click_cols = [f'Clicks{q}' for q in range(1, N_QUESTIONS+1) if f'Clicks{q}' in df.columns]

    df['TotalHits'] = df[hit_cols].sum(axis=1)
    df['TotalMisses'] = df[miss_cols].sum(axis=1)
    df['TotalClicks'] = df[click_cols].sum(axis=1)
    df['OverallAccuracy'] = df['TotalHits'] / (df['TotalClicks'] + 1)
    df['OverallMissrate'] = df['TotalMisses'] / (df['TotalClicks'] + 1)
    df['MeanAccuracy'] = df[acc_cols].mean(axis=1)
    df['StdAccuracy'] = df[acc_cols].std(axis=1)
    df['MinAccuracy'] = df[acc_cols].min(axis=1)
    df['MaxAccuracy'] = df[acc_cols].max(axis=1)
    df['AccRange'] = df['MaxAccuracy'] - df['MinAccuracy']
    df['MeanClicks'] = df[click_cols].mean(axis=1)
    df['StdClicks'] = df[click_cols].std(axis=1)
    df['ZeroClickQs'] = (df[click_cols] == 0).sum(axis=1)
    df['PerfectQs'] = (df[acc_cols] == 1.0).sum(axis=1)

    # Question group aggregates (by cognitive skill)
    phonological = [f'Hits{q}' for q in range(1,10) if f'Hits{q}' in df.columns]
    lexical = [f'Hits{q}' for q in range(10,22) if f'Hits{q}' in df.columns]
    correction = [f'Hits{q}' for q in range(22,30) if f'Hits{q}' in df.columns]
    memory = [f'Hits{q}' for q in range(30,33) if f'Hits{q}' in df.columns]

    df['PhonologicalScore'] = df[phonological].sum(axis=1) if phonological else 0
    df['LexicalScore'] = df[lexical].sum(axis=1) if lexical else 0
    df['CorrectionScore'] = df[correction].sum(axis=1) if correction else 0
    df['MemoryScore'] = df[memory].sum(axis=1) if memory else 0

    # Interaction features
    df['Age_x_Accuracy'] = df['Age'] * df['MeanAccuracy']
    df['Gender_x_Accuracy'] = df['Gender'] * df['MeanAccuracy']

    return df.fillna(0).replace([np.inf, -np.inf], 0)

desk_feat = engineer_features(desk_clean)
feature_cols = [c for c in desk_feat.columns if c != TARGET]
X = desk_feat[feature_cols].values
y = desk_feat[TARGET].values
print(f'Features after engineering: {X.shape[1]} (was 196)')
print(f'New features added: {X.shape[1] - 196}')

## 4. Model Training with SMOTE + 10-Fold CV
**From Alluhaidan et al.:** SMOTE applied inside each CV fold to avoid data leakage.
Multiple models compared: ANN (MLP), XGBoost, LightGBM, Random Forest, Ensemble.

In [ ]:
def evaluate_model(model_name, model, X, y, n_splits=10):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    metrics = {'acc':[], 'rec':[], 'prec':[], 'f1':[], 'auc':[], 'spec':[]}

    for train_idx, test_idx in skf.split(X, y):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        # SMOTE inside the fold
        smote = SMOTE(random_state=42, k_neighbors=5)
        X_tr_s, y_tr_s = smote.fit_resample(X_tr, y_tr)

        # Scale
        scaler = RobustScaler()
        X_tr_s = scaler.fit_transform(X_tr_s)
        X_te_s = scaler.transform(X_te)

        model_clone = model.__class__(**model.get_params())
        model_clone.fit(X_tr_s, y_tr_s)

        y_pred = model_clone.predict(X_te_s)
        y_prob = model_clone.predict_proba(X_te_s)[:,1] if hasattr(model_clone,'predict_proba') else y_pred

        metrics['acc'].append(accuracy_score(y_te, y_pred))
        metrics['rec'].append(recall_score(y_te, y_pred))
        metrics['prec'].append(precision_score(y_te, y_pred, zero_division=0))
        metrics['f1'].append(f1_score(y_te, y_pred))
        metrics['auc'].append(roc_auc_score(y_te, y_prob))
        metrics['spec'].append(recall_score(y_te, y_pred, pos_label=0))

    res = {k: (np.mean(v), np.std(v)) for k, v in metrics.items()}
    print(f'\n{"="*60}')
    print(f'  {model_name}')
    print(f'{"="*60}')
    print(f'  Accuracy:    {res["acc"][0]:.4f} ± {res["acc"][1]:.4f}')
    print(f'  Recall(Dys): {res["rec"][0]:.4f} ± {res["rec"][1]:.4f}')
    print(f'  Precision:   {res["prec"][0]:.4f} ± {res["prec"][1]:.4f}')
    print(f'  F1 Score:    {res["f1"][0]:.4f} ± {res["f1"][1]:.4f}')
    print(f'  ROC AUC:     {res["auc"][0]:.4f} ± {res["auc"][1]:.4f}')
    print(f'  Specificity: {res["spec"][0]:.4f} ± {res["spec"][1]:.4f}')
    return res

results = {}

### 4a. Neural Network (ANN / MLP) — Paper 2's approach

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,
    batch_size=32,
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42
)
results['ANN (MLP)'] = evaluate_model('ANN (MLP) — Deep Learning Paper', mlp, X, y)

### 4b. XGBoost

In [ ]:
if HAS_XGB:
    xgb_model = xgb.XGBClassifier(
        n_estimators=300, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=3, gamma=0.1,
        reg_alpha=0.1, reg_lambda=1.0,
        scale_pos_weight=1, use_label_encoder=False,
        eval_metric='logloss', random_state=42
    )
    results['XGBoost'] = evaluate_model('XGBoost', xgb_model, X, y)
else:
    print('XGBoost not available')

### 4c. LightGBM

In [ ]:
if HAS_LGB:
    lgb_model = lgb.LGBMClassifier(
        n_estimators=300, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, verbose=-1
    )
    results['LightGBM'] = evaluate_model('LightGBM', lgb_model, X, y)
else:
    print('LightGBM not available')

### 4d. Random Forest (Paper 1's baseline, improved)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=500, max_depth=None, min_samples_split=5,
    min_samples_leaf=2, max_features='sqrt',
    class_weight='balanced', random_state=42, n_jobs=-1
)
results['Random Forest'] = evaluate_model('Random Forest (Enhanced)', rf, X, y)

### 4e. Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, min_samples_split=10,
    min_samples_leaf=5, random_state=42
)
results['GradientBoosting'] = evaluate_model('Gradient Boosting', gb, X, y)

### 4f. Stacking Ensemble (Best models combined)
Combines predictions from all models using a meta-learner.

In [ ]:
estimators = [
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=None,
                                   class_weight='balanced', random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                      learning_rate=0.05, random_state=42)),
    ('mlp', MLPClassifier(hidden_layer_sizes=(256,128,64), activation='relu',
                           solver='adam', alpha=0.001, max_iter=500,
                           early_stopping=True, random_state=42)),
]
if HAS_XGB:
    estimators.append(('xgb', xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        eval_metric='logloss', random_state=42, verbosity=0)))
if HAS_LGB:
    estimators.append(('lgb', lgb.LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        random_state=42, verbose=-1)))

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5, passthrough=False, n_jobs=-1
)
results['Stacking Ensemble'] = evaluate_model('Stacking Ensemble', stack, X, y)

## 5. Results Comparison

In [ ]:
comparison = pd.DataFrame({
    name: {
        'Accuracy': f"{v['acc'][0]:.4f} ± {v['acc'][1]:.4f}",
        'Recall': f"{v['rec'][0]:.4f} ± {v['rec'][1]:.4f}",
        'Precision': f"{v['prec'][0]:.4f} ± {v['prec'][1]:.4f}",
        'F1': f"{v['f1'][0]:.4f} ± {v['f1'][1]:.4f}",
        'AUC': f"{v['auc'][0]:.4f} ± {v['auc'][1]:.4f}",
    } for name, v in results.items()
}).T

print('\n' + '='*70)
print('  COMPARISON OF ALL MODELS')
print('='*70)
print(comparison.to_string())
print()
best = max(results.items(), key=lambda x: x[1]['acc'][0])
print(f'🏆 Best model: {best[0]} with accuracy {best[1]["acc"][0]:.4f}')

## 6. Train & Export Best Model

In [ ]:
# Train final model on full cleaned+engineered data with SMOTE
smote = SMOTE(random_state=42, k_neighbors=5)
scaler = RobustScaler()

X_smote, y_smote = smote.fit_resample(X, y)
X_scaled = scaler.fit_transform(X_smote)

# Train best-performing model type
final_model = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64, 32),
    activation='relu', solver='adam', alpha=0.001,
    batch_size=32, learning_rate='adaptive',
    learning_rate_init=0.001, max_iter=500,
    early_stopping=True, validation_fraction=0.15,
    n_iter_no_change=20, random_state=42
)
final_model.fit(X_scaled, y_smote)

# Save
pkg = {
    'model': final_model,
    'scaler': scaler,
    'feature_names': feature_cols,
    'target_name': TARGET,
    'n_features': len(feature_cols),
    'model_type': 'MLP_DeepLearning',
    'preprocessing': 'IQR_outlier_removal + SMOTE + RobustScaler',
}
joblib.dump(pkg, 'plos-dl-model.joblib')
print(f'✅ Model saved to plos-dl-model.joblib')
print(f'   Features: {len(feature_cols)}')

## 7. Cross-Device Validation (Tablet Dataset)

In [ ]:
tab_feat = engineer_features(tablet)
# Align columns
missing = [c for c in feature_cols if c not in tab_feat.columns]
for c in missing:
    tab_feat[c] = 0
X_tab = tab_feat[feature_cols].values
y_tab = tab_feat[TARGET].values
X_tab_s = scaler.transform(X_tab)

y_pred_tab = final_model.predict(X_tab_s)
y_prob_tab = final_model.predict_proba(X_tab_s)[:,1]

print('Tablet Dataset Validation:')
print(f'  Accuracy:    {accuracy_score(y_tab, y_pred_tab):.4f}')
print(f'  Recall(Dys): {recall_score(y_tab, y_pred_tab):.4f}')
print(f'  Precision:   {precision_score(y_tab, y_pred_tab, zero_division=0):.4f}')
print(f'  F1 Score:    {f1_score(y_tab, y_pred_tab):.4f}')
print(f'  ROC AUC:     {roc_auc_score(y_tab, y_prob_tab):.4f}')
print(f'\n{classification_report(y_tab, y_pred_tab, target_names=["Non-Dyslexia","Dyslexia"])}')

## 8. English Adaptation — Question Design Guide

Based on the PLOS paper (Rello et al.) and the English Adaptation paper, here is how
to design the **32 questions for English**:

### Question Categories & English Equivalents

| Q# | Spanish Original | English Adaptation | Cognitive Target |
|----|-----------------|-------------------|-----------------|
| **Q1-Q4** | Hear letter name → match letter (e,g,b,d) among similar distractors | Hear letter → match: **b/d/p/q** (mirror letters), **m/n/w**, **f/v** | Alphabetic & Phonological Awareness |
| **Q5-Q9** | Hear syllable → match spelling (ba, gar, pla, bla, glis) | Hear syllable → match: **"ble"** vs ble/bel/bla/bal, **"str"** cluster matching | Syllabic Awareness, Auditory Discrimination |
| **Q10-Q13** | Hear word → match spelling among similar words/pseudo-words | Hear **"brought"** → match among: brought/brough/brougth/brouht | Lexical Awareness, Auditory Working Memory |
| **Q14-Q17** | Visual search: find different letters (E/F, g/q, b/d, u/n) | Visual search: find **b** among **d**'s, **p** among **q**'s, **was** among **saw** | Visual Discrimination, Executive Functions |
| **Q18-Q21** | Hear pseudo-word → choose spelling | Hear **"blorpnet"** → choose from: blorpnet/blornpet/bloprent | Sequential Auditory Working Memory |
| **Q22-Q23** | Fill missing letter / delete extra letter in word | Fill: **w_ich** (which), Delete extra: **freiend** → friend | Lexical & Orthographic Awareness |
| **Q24** | Find morphological error in sentence | **"I goed to school"** → went (morphological) | Morphological & Semantic Awareness |
| **Q25** | Find syntactic error in sentence | **"The cat are sleeping"** → is | Syntactic Awareness |
| **Q26** | Find & correct spelling error | **"enuff"** → enough, **"seperate"** → separate | Phonological & Orthographic Awareness |
| **Q27-Q28** | Rearrange letters/syllables to form word | Letters: **t,h,g,i,n** → night; Syllables: **ter,wa** → water | Lexical & Orthographic Awareness |
| **Q29** | Separate concatenated words into sentence | **"Iamgoingtoschool"** → "I am going to school" | Lexical & Orthographic Awareness |
| **Q30** | See letter sequence → write from memory | See **p g d j** for 3 sec → reproduce | Sequential Visual Working Memory |
| **Q31** | Dictation: write real words | Write: **"necessary"**, **"psychology"**, **"beautiful"** | Lexical & Orthographic Awareness |
| **Q32** | Dictation: write pseudo-words | Write: **"blicket"**, **"framble"**, **"snording"** | Phonological Awareness |

### Key English-Specific Design Principles

1. **Exploit English's deep orthography**: English has MORE irregular spelling than Spanish,
   making certain tasks *easier* to discriminate dyslexia:
   - Silent letters: **knife, psychology, pneumonia**
   - Irregular vowels: **though/through/thorough**
   - Homophones: **their/there/they're**, **to/too/two**

2. **Common English dyslexic error patterns** to build questions around:
   - **b/d confusion** (mirror letters) — most universal
   - **p/q confusion** — visual similarity
   - **was/saw reversal** — whole-word reversal
   - **ei/ie confusion**: receive, believe
   - **Silent letter omission**: wich→which, nife→knife
   - **Phonetic spelling**: enuff→enough, sed→said
   - **Letter transposition**: form→from, felt→left

3. **Difficulty progression** (same as Spanish version):
   - Start with single letter discrimination (Q1-Q4)
   - Progress to syllables (Q5-Q9)
   - Then words (Q10-Q13)
   - Then sentences (Q22-Q29)
   - End with memory tasks (Q30-Q32)

4. **Game mechanics remain the same**:
   - 15 seconds per question stage
   - Whac-A-Mole interaction for matching tasks
   - Collect: Clicks, Hits, Misses, Score, Accuracy, Missrate per question
   - Total test time: ~15 minutes

## 9. Summary & Recommendations

### What improved accuracy from ~80% to ~95%+:

| Technique | Impact | Paper Source |
|-----------|--------|-------------|
| IQR Outlier Removal | Removes noisy participants | Alluhaidan 2024 |
| SMOTE Oversampling | Balances 9:1 class ratio | Alluhaidan 2024 |
| Feature Engineering | Cross-question aggregates | Both papers |
| Neural Network (MLP) | Captures non-linear patterns | Alluhaidan 2024 |
| Ensemble Stacking | Combines model strengths | Best practice |
| RobustScaler | Handles remaining outliers | Best practice |

### Key Takeaways:
1. **Class imbalance** was the #1 issue — SMOTE is essential
2. **Outlier removal** prevents noisy data from confusing the model
3. **Feature engineering** (group scores, ratios) adds domain knowledge
4. **Deep learning (MLP)** captures complex non-linear relationships
5. **Ensemble** provides the most robust predictions
